In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("ExpenseMonitoringSystem").getOrCreate()

In [3]:
# Load users dataset
users_df=spark.read.csv("users.csv",header=True,inferSchema=True)
# Load expenses dataset
expenses_df=spark.read.csv("cleaned_expenses.csv",header=True,inferSchema=True)
expenses_df=expenses_df.withColumn("month",date_format(col("expense_date"),"yyyy-MM"))

In [4]:
# Monthly user spending
monthly_spend=expenses_df.groupBy("user_id","user_name","month").agg(sum("amount").alias("monthly_spend"))
print("Monthly user spending")
monthly_spend.show()

Monthly user spending
+-------+---------------+-------+-------------+
|user_id|      user_name|  month|monthly_spend|
+-------+---------------+-------+-------------+
|      3|Sonakshi Sharma|2026-04|       2500.0|
|      3|Sonakshi Sharma|2026-05|      12000.0|
|      2|   Ankit Sharma|2026-04|       8000.0|
|      2|   Ankit Sharma|2026-05|        199.0|
|      1|Priyanka Sharma|2026-03|       1650.0|
|      4|   Karan Sharma|2026-05|       5000.0|
+-------+---------------+-------+-------------+



In [5]:
# Join datasets
final_df=monthly_spend.join(users_df,on="user_id",how="inner")

# Calculate savings
final_df=final_df.withColumn("savings",col("monthly_income")-col("monthly_spend"))

# Generate alerts
final_df=final_df.withColumn("alert",when(col("savings")<10000,"High Spending").otherwise("Normal"))
print("Final ETL Report")
final_df.show()

Final ETL Report
+-------+---------------+-------+-------------+---------------+--------------------+----------+------+--------------+-------+------+
|user_id|      user_name|  month|monthly_spend|      user_name|               email|     phone|gender|monthly_income|savings| alert|
+-------+---------------+-------+-------------+---------------+--------------------+----------+------+--------------+-------+------+
|      3|Sonakshi Sharma|2026-04|       2500.0|Sonakshi Sharma|sonakshisharma@gm...|9876543212|Female|         60000|57500.0|Normal|
|      3|Sonakshi Sharma|2026-05|      12000.0|Sonakshi Sharma|sonakshisharma@gm...|9876543212|Female|         60000|48000.0|Normal|
|      2|   Ankit Sharma|2026-04|       8000.0|   Ankit Sharma|ankitsharma@gmail...|9876543211|  Male|         70000|62000.0|Normal|
|      2|   Ankit Sharma|2026-05|        199.0|   Ankit Sharma|ankitsharma@gmail...|9876543211|  Male|         70000|69801.0|Normal|
|      1|Priyanka Sharma|2026-03|       1650.0|Priya

In [7]:
# My azure credits got over. I ran this in google colab instead of databricks. So I saved the result as a csv file instead of delta
# Save as Delta Table
# final_df.write.format("delta").mode("overwrite").saveAsTable("final_report")

# Save CSV report
final_df.toPandas().to_csv("final_report.csv",index=False)
print("Final report saved successfully")

Final report saved successfully
